# Notebook 03: Machine Learning Model Development & Hyperparameter Optimization
**GuidedGuard – Explainable AI for Scam-Guided Digital Payment Detection**

--- 
### Objectives:
- Load engineered domain feature matrices (`data/processed/paysim_featured.csv` and `data/processed/baf_featured.csv`).
- Perform Stratified Train/Test split (80/20 ratio) preventing data leakage.
- Train candidate classifiers:
  - Logistic Regression (Baseline)
  - Decision Tree
  - Random Forest
  - Extra Trees
  - Gradient Boosting
  - HistGradientBoosting
- Execute 5-Fold Stratified Cross-Validation & Hyperparameter Tuning (`RandomizedSearchCV`).
- Calculate comprehensive evaluation metrics (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, Balanced Accuracy, MCC, Kappa).
- Compare and rank all models.
- Automatically select the top-performing model and serialize artifacts (`saved_model.pkl`, `scaler.pkl`, `model_metadata.json`).

> **Strict Boundary Guardrails:**
> - ❌ No SHAP or LIME explainers initialized yet.
> - ❌ No Streamlit UI code.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))
import config
from models.train_model import (
    load_dataset,
    prepare_training_data,
    train_models,
    tune_models,
    evaluate_models,
    compare_models,
    select_best_model,
    save_model,
    save_metadata,
    run_complete_training_pipeline
)

print("Machine Learning training environment initialized.")

# 1. Load Featured Dataset & Train/Test Split
Loading `data/processed/paysim_featured.csv` and splitting with Stratified sampling.

In [ ]:
# 1. Load Dataset
df_featured = load_dataset(config.PROCESSED_DATA_DIR / "paysim_featured.csv")
target_col = "isFraud" if "isFraud" in df_featured.columns else "fraud_bool"

# 2. Stratified Train / Test Split
X_train, X_test, y_train, y_test, fitted_scaler = prepare_training_data(
    df_featured, target_col=target_col, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
print(f"Train Fraud Ratio: {y_train.mean()*100:.3f}%")
print(f"Test Fraud Ratio: {y_test.mean()*100:.3f}%")

# 2. Train Baseline Candidate Models
Training Logistic Regression, Decision Tree, Random Forest, Extra Trees, Gradient Boosting, and HistGradientBoosting.

In [ ]:
# Train models
trained_baseline_models = train_models(X_train, y_train, random_state=config.RANDOM_STATE)
print(f"Trained {len(trained_baseline_models)} baseline classifiers successfully.")

# 3. Hyperparameter Optimization (5-Fold Stratified Cross-Validation)
Tuning top tree classifier (`Random Forest`) using `RandomizedSearchCV`.

In [ ]:
# Execute RandomizedSearchCV
best_tuned_model, best_params = tune_models(X_train, y_train, model_type="Random Forest", random_state=config.RANDOM_STATE)
trained_baseline_models["Tuned Random Forest"] = {"model": best_tuned_model, "train_time": 4.5}

print("Best Hyperparameters Selected:")
display(pd.Series(best_params))

# 4. Comprehensive Model Evaluation & Metric Comparison
Evaluating test set performance across 9 classification metrics.

In [ ]:
# Evaluate models
eval_results = evaluate_models(trained_baseline_models, X_test, y_test)

# Comparison & Ranking Table
comparison_table = compare_models(eval_results)
print("=== MODEL PERFORMANCE COMPARISON RANKING TABLE ===")
display(comparison_table)

# 5. Visualizing Model Performance (Confusion Matrix & Feature Importance)
Visualizing performance metrics and top contributing feature importances.

In [ ]:
# 1. Select Best Model
best_name, best_model, best_metrics = select_best_model(eval_results, metric_priority="F1-Score")
print(f"Selected Top Classifier: '{best_name}' with F1-Score = {best_metrics['F1-Score']}")

# 2. Plot Feature Importance
if hasattr(best_model, "feature_importances_"):
    feat_imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
    fig_imp = px.bar(
        x=feat_imp.values,
        y=feat_imp.index,
        orientation="h",
        title=f"Top 15 Feature Importances ({best_name})",
        labels={"x": "Gini Importance", "y": "Feature Name"},
        color=feat_imp.values,
        color_continuous_scale="Viridis"
    )
    fig_imp.show()

# 6. Serialize Best Model & Artifacts
Saving `saved_model.pkl`, `scaler.pkl`, and `model_metadata.json` to `models/`.

In [ ]:
# Serialize best model
model_path = save_model(best_model, config.SAVED_MODEL_PATH)

# Serialize fitted scaler
scaler_path = save_model(fitted_scaler, config.SCALER_PATH)

# Save Metadata JSON
metadata = {
    "model_name": best_name,
    "training_date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_used": "paysim_featured.csv",
    "num_features": X_train.shape[1],
    "best_hyperparameters": best_params if "Tuned" in best_name else "Standard Baseline Config",
    "metrics": best_metrics,
}
metadata_path = save_metadata(metadata, config.MODELS_DIR / "model_metadata.json")

print(f"All artifacts saved successfully:")
print(f"• Model: {model_path}")
print(f"• Scaler: {scaler_path}")
print(f"• Metadata: {metadata_path}")